# Massachusetts 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Massachusetts, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals). The two datasets only have town-level vote counts, without associated county. Thus, a manual crosswalk was performed later.

**Output**: A single CSV where each row is a county and columns include:

- Primary per-candidate vote counts (prefixed with `pri_`)
- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_primary_total`, `dem_primary_total`, `grn_primary_total`, `rep_general_total`, `dem_general_total`, `lib_general_total`, `cst_general_total`, `grn_general_total`, `ind_general_total`

**Last Updated**: 2025/10/22

## 0. Library Import

In [69]:
import re 
import pandas as pd
import numpy as numpy
from pathlib import Path

## 1. Inputs & Parameters

In [70]:
# MA 2008 dataset path
PRIMARY_PATH = r"../../data/raw/2008/MA/20080205__ma__primary__president__precinct.csv"
GENERAL_PATH = r"../../data/raw/2008/MA/20081104__ma__general__precinct.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/MA/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### a. Primary Election Dataset

In [71]:
# Load primary data
primary_df = pd.read_csv(PRIMARY_PATH)
primary_df.head(DISPLAY_ROWS)

,town,ward,precinct,office,district,party,candidate,votes
0,Watertown,NaN,10,President,NaN,Green-Rainbow,Ralph Nader,0
1,Marlborough,4,1,President,NaN,Green-Rainbow,Kat Swift,0
2,Andover,NaN,3,President,NaN,Democratic,Bill Richardson,2
3,Wellfleet,NaN,1,President,NaN,Republican,Ron Paul,8
4,Cambridge,1,3,President,NaN,Republican,John McCain,37
5,Saugus,NaN,8,President,NaN,Republican,No Preference,2
6,Cambridge,2,3,President,NaN,Green-Rainbow,Kat Swift,0
7,Malden,5,1,President,NaN,Republican,Blank Votes,0
8,Norton,NaN,2,President,NaN,Republican,Total Votes Cast,428
9,Cambridge,1,2,President,NaN,Democratic,Christopher J. Dodd,1


In [72]:
# Different values in 'office' column
primary_df["office"].value_counts()

office
President    73092
Name: count, dtype: int64

In [73]:
# Number of missing values in each column
primary_df.isna().sum()

town             0
ward         37392
precinct        34
office           0
district     73092
party            2
candidate        0
votes            0
dtype: int64

In [74]:
# Now, drop the "office" column since it has a single value
# Also, drop the district column since it's all missing values
primary_df = primary_df.drop(columns=["office", "district"]).reset_index(drop=True)
primary_df.head(DISPLAY_ROWS)

,town,ward,precinct,party,candidate,votes
0,Watertown,NaN,10,Green-Rainbow,Ralph Nader,0
1,Marlborough,4,1,Green-Rainbow,Kat Swift,0
2,Andover,NaN,3,Democratic,Bill Richardson,2
3,Wellfleet,NaN,1,Republican,Ron Paul,8
4,Cambridge,1,3,Republican,John McCain,37
5,Saugus,NaN,8,Republican,No Preference,2
6,Cambridge,2,3,Green-Rainbow,Kat Swift,0
7,Malden,5,1,Republican,Blank Votes,0
8,Norton,NaN,2,Republican,Total Votes Cast,428
9,Cambridge,1,2,Democratic,Christopher J. Dodd,1


Since we want county-level vote counts, we first groupby town to get the town-level vote count, then we will perform a cross-walk to map from town to county.

In [75]:
# Calculate town-level vote count
primary_df = (
    primary_df.groupby(['town', 'candidate', 'party'], as_index=False)['votes'].sum()
)

primary_df.head(DISPLAY_ROWS)

,town,candidate,party,votes
0,Abington,All Others,Democratic,5
1,Abington,All Others,Green-Rainbow,0
2,Abington,All Others,Republican,2
3,Abington,Barack Obama,Democratic,824
4,Abington,Bill Richardson,Democratic,8
5,Abington,Blank Votes,Democratic,10
6,Abington,Blank Votes,Green-Rainbow,0
7,Abington,Blank Votes,Republican,1
8,Abington,Christopher J. Dodd,Democratic,1
9,Abington,Cynthia McKinney,Green-Rainbow,0


In [76]:
town_to_county = {
    # Barnstable County
    "Barnstable": "Barnstable", "Bourne": "Barnstable", "Brewster": "Barnstable",
    "Chatham": "Barnstable", "Dennis": "Barnstable", "Eastham": "Barnstable",
    "Falmouth": "Barnstable", "Harwich": "Barnstable", "Mashpee": "Barnstable",
    "Orleans": "Barnstable", "Provincetown": "Barnstable", "Sandwich": "Barnstable",
    "Truro": "Barnstable", "Wellfleet": "Barnstable", "Yarmouth": "Barnstable",

    # Berkshire County
    "Adams": "Berkshire", "Alford": "Berkshire", "Becket": "Berkshire",
    "Cheshire": "Berkshire", "Clarksburg": "Berkshire", "Dalton": "Berkshire",
    "Egremont": "Berkshire", "Florida": "Berkshire", "Great Barrington": "Berkshire",
    "Hancock": "Berkshire", "Hinsdale": "Berkshire", "Lanesborough": "Berkshire",
    "Lee": "Berkshire", "Lenox": "Berkshire", "Monterey": "Berkshire",
    "Mount Washington": "Berkshire", "New Ashford": "Berkshire",
    "New Marlborough": "Berkshire", "North Adams": "Berkshire", "N. Adams": "Berkshire",
    "Otis": "Berkshire", "Peru": "Berkshire", "Pittsfield": "Berkshire",
    "Richmond": "Berkshire", "Sandisfield": "Berkshire", "Savoy": "Berkshire",
    "Sheffield": "Berkshire", "Stockbridge": "Berkshire", "Tyringham": "Berkshire",
    "Washington": "Berkshire", "West Stockbridge": "Berkshire", "W. Stockbridge": "Berkshire",
    "Williamstown": "Berkshire", "Windsor": "Berkshire",

    # Bristol County
    "Attleboro": "Bristol", "Berkley": "Bristol", "Dartmouth": "Bristol",
    "Dighton": "Bristol", "Easton": "Bristol", "Fall River": "Bristol",
    "Fairhaven": "Bristol", "Freetown": "Bristol", "Mansfield": "Bristol",
    "New Bedford": "Bristol", "Norton": "Bristol", "North Attleborough": "Bristol",
    "N. Attleborough": "Bristol", "Raynham": "Bristol", "Rehoboth": "Bristol",
    "Seekonk": "Bristol", "Somerset": "Bristol", "Swansea": "Bristol",
    "Taunton": "Bristol", "Westport": "Bristol", "Acushnet": "Bristol",

    # Dukes County
    "Aquinnah": "Dukes", "Chilmark": "Dukes", "Edgartown": "Dukes",
    "Gosnold": "Dukes", "Oak Bluffs": "Dukes", "Tisbury": "Dukes",
    "West Tisbury": "Dukes", "W. Tisbury": "Dukes",

    # Essex County
    "Amesbury": "Essex", "Andover": "Essex", "Beverly": "Essex",
    "Boxford": "Essex", "Danvers": "Essex", "Essex": "Essex",
    "Georgetown": "Essex", "Gloucester": "Essex", "Groveland": "Essex",
    "Hamilton": "Essex", "Haverhill": "Essex", "Ipswich": "Essex",
    "Lawrence": "Essex", "Lynn": "Essex", "Lynnfield": "Essex",
    "Manchester-by-the-Sea": "Essex", "Marblehead": "Essex", "Merrimac": "Essex",
    "Methuen": "Essex", "Middleton": "Essex", "Nahant": "Essex",
    "Newbury": "Essex", "Newburyport": "Essex", "North Andover": "Essex",
    "N. Andover": "Essex", "Peabody": "Essex", "Rockport": "Essex",
    "Rowley": "Essex", "Salem": "Essex", "Salisbury": "Essex",
    "Saugus": "Essex", "Swampscott": "Essex", "Topsfield": "Essex",
    "West Newbury": "Essex", "W. Newbury": "Essex", "Wenham": "Essex",

    # Franklin County
    "Ashfield": "Franklin", "Bernardston": "Franklin", "Buckland": "Franklin",
    "Charlemont": "Franklin", "Colrain": "Franklin", "Conway": "Franklin",
    "Deerfield": "Franklin", "Erving": "Franklin", "Gill": "Franklin",
    "Greenfield": "Franklin", "Hawley": "Franklin", "Heath": "Franklin",
    "Leverett": "Franklin", "Leyden": "Franklin", "Monroe": "Franklin",
    "Montague": "Franklin", "New Salem": "Franklin", "Northfield": "Franklin",
    "Orange": "Franklin", "Rowe": "Franklin", "Shelburne": "Franklin",
    "Shutesbury": "Franklin", "Sunderland": "Franklin", "Warwick": "Franklin",
    "Wendell": "Franklin", "Whately": "Franklin",

    # Hampden County
    "Agawam": "Hampden", "Blandford": "Hampden", "Brimfield": "Hampden",
    "Chester": "Hampden", "Chicopee": "Hampden", "East Longmeadow": "Hampden",
    "E. Longmeadow": "Hampden", "Granville": "Hampden", "Hampden": "Hampden",
    "Holyoke": "Hampden", "Longmeadow": "Hampden", "Ludlow": "Hampden",
    "Monson": "Hampden", "Montgomery": "Hampden", "Palmer": "Hampden",
    "Russell": "Hampden", "Southwick": "Hampden", "Springfield": "Hampden",
    "Tolland": "Hampden", "Wales": "Hampden", "West Springfield": "Hampden",
    "W. Springfield": "Hampden", "Westfield": "Hampden", "Wilbraham": "Hampden",
    "Holland": "Hampden",

    # Hampshire County
    "Amherst": "Hampshire", "Belchertown": "Hampshire", "Chesterfield": "Hampshire",
    "Cummington": "Hampshire", "Easthampton": "Hampshire", "Goshen": "Hampshire",
    "Granby": "Hampshire", "Hadley": "Hampshire", "Hatfield": "Hampshire",
    "Huntington": "Hampshire", "Middlefield": "Hampshire", "Northampton": "Hampshire",
    "Pelham": "Hampshire", "Plainfield": "Hampshire", "South Hadley": "Hampshire",
    "S. Hadley": "Hampshire", "Southampton": "Hampshire", "Ware": "Hampshire",
    "Westhampton": "Hampshire", "Williamsburg": "Hampshire", "Worthington": "Hampshire",

    # Middlesex County
    "Acton": "Middlesex", "Arlington": "Middlesex", "Ashby": "Middlesex",
    "Ashland": "Middlesex", "Ayer": "Middlesex", "Bedford": "Middlesex",
    "Belmont": "Middlesex", "Billerica": "Middlesex", "Boxborough": "Middlesex",
    "Burlington": "Middlesex", "Cambridge": "Middlesex", "Carlisle": "Middlesex",
    "Chelmsford": "Middlesex", "Concord": "Middlesex", "Dracut": "Middlesex",
    "Dunstable": "Middlesex", "Everett": "Middlesex", "Framingham": "Middlesex",
    "Groton": "Middlesex", "Holliston": "Middlesex", "Hopkinton": "Middlesex",
    "Hudson": "Middlesex", "Lexington": "Middlesex", "Lincoln": "Middlesex",
    "Littleton": "Middlesex", "Lowell": "Middlesex", "Malden": "Middlesex",
    "Marlborough": "Middlesex", "Maynard": "Middlesex", "Medford": "Middlesex",
    "Melrose": "Middlesex", "Natick": "Middlesex", "Newton": "Middlesex",
    "North Reading": "Middlesex", "N. Reading": "Middlesex", "Pepperell": "Middlesex",
    "Reading": "Middlesex", "Sherborn": "Middlesex", "Shirley": "Middlesex",
    "Somerville": "Middlesex", "Stoneham": "Middlesex", "Stow": "Middlesex",
    "Sudbury": "Middlesex", "Tewksbury": "Middlesex", "Townsend": "Middlesex",
    "Tyngsborough": "Middlesex", "Wakefield": "Middlesex", "Waltham": "Middlesex",
    "Watertown": "Middlesex", "Wayland": "Middlesex", "Westford": "Middlesex",
    "Weston": "Middlesex", "Wilmington": "Middlesex", "Winchester": "Middlesex",
    "Woburn": "Middlesex",

    # Nantucket County
    "Nantucket": "Nantucket",

    # Norfolk County
    "Avon": "Norfolk", "Bellingham": "Norfolk", "Braintree": "Norfolk",
    "Brookline": "Norfolk", "Canton": "Norfolk", "Cohasset": "Norfolk",
    "Dedham": "Norfolk", "Dover": "Norfolk", "Foxborough": "Norfolk",
    "Franklin": "Norfolk", "Holbrook": "Norfolk", "Medfield": "Norfolk",
    "Medway": "Norfolk", "Milton": "Norfolk", "Needham": "Norfolk",
    "Norfolk": "Norfolk", "Norwood": "Norfolk", "Plainville": "Norfolk",
    "Quincy": "Norfolk", "Randolph": "Norfolk", "Sharon": "Norfolk",
    "Stoughton": "Norfolk", "Walpole": "Norfolk", "Wellesley": "Norfolk",
    "Westwood": "Norfolk", "Weymouth": "Norfolk", "Wrentham": "Norfolk",
    "Millis": "Norfolk",

    # Plymouth County
    "Abington": "Plymouth", "Bridgewater": "Plymouth",
    "East Bridgewater": "Plymouth", "E. Bridgewater": "Plymouth",
    "West Bridgewater": "Plymouth", "W. Bridgewater": "Plymouth",
    "Brockton": "Plymouth", "Carver": "Plymouth", "Duxbury": "Plymouth",
    "Halifax": "Plymouth", "Hanover": "Plymouth", "Hanson": "Plymouth",
    "Hingham": "Plymouth", "Hull": "Plymouth", "Kingston": "Plymouth",
    "Lakeville": "Plymouth", "Marion": "Plymouth", "Marshfield": "Plymouth",
    "Mattapoisett": "Plymouth", "Middleborough": "Plymouth",
    "Norwell": "Plymouth", "Pembroke": "Plymouth", "Plymouth": "Plymouth",
    "Plympton": "Plymouth", "Rochester": "Plymouth", "Rockland": "Plymouth",
    "Scituate": "Plymouth", "Wareham": "Plymouth", "Whitman": "Plymouth",

    # Suffolk County
    "Boston": "Suffolk", "Chelsea": "Suffolk", "Revere": "Suffolk", "Winthrop": "Suffolk",

    # Worcester County
    "Ashburnham": "Worcester", "Athol": "Worcester", "Auburn": "Worcester",
    "Barre": "Worcester", "Berlin": "Worcester", "Blackstone": "Worcester",
    "Bolton": "Worcester", "Boylston": "Worcester", "Brookfield": "Worcester",
    "Charlton": "Worcester", "Clinton": "Worcester", "Douglas": "Worcester",
    "Dudley": "Worcester", "East Brookfield": "Worcester", "E. Brookfield": "Worcester",
    "Fitchburg": "Worcester", "Gardner": "Worcester", "Grafton": "Worcester",
    "Hardwick": "Worcester", "Harvard": "Worcester", "Holden": "Worcester",
    "Hopedale": "Worcester", "Hubbardston": "Worcester", "Lancaster": "Worcester",
    "Leicester": "Worcester", "Leominster": "Worcester", "Lunenburg": "Worcester",
    "Mendon": "Worcester", "Milford": "Worcester", "Millbury": "Worcester",
    "Millville": "Worcester", "New Braintree": "Worcester", "Northborough": "Worcester",
    "North Brookfield": "Worcester", "N. Brookfield": "Worcester",
    "Northbridge": "Worcester", "Oakham": "Worcester", "Oxford": "Worcester",
    "Paxton": "Worcester", "Petersham": "Worcester", "Phillipston": "Worcester",
    "Princeton": "Worcester", "Royalston": "Worcester", "Rutland": "Worcester",
    "Shrewsbury": "Worcester", "Southborough": "Worcester", "Southbridge": "Worcester",
    "Spencer": "Worcester", "Sterling": "Worcester", "Sturbridge": "Worcester",
    "Sutton": "Worcester", "Templeton": "Worcester", "Upton": "Worcester",
    "Uxbridge": "Worcester", "Warren": "Worcester", "Webster": "Worcester",
    "West Boylston": "Worcester", "W. Boylston": "Worcester",
    "West Brookfield": "Worcester", "W. Brookfield": "Worcester",
    "Westminster": "Worcester", "Winchendon": "Worcester", "Worcester": "Worcester", 
    "Westborough": "Worcester",

    # Extras / abbreviations from your list
    "Cambridge": "Middlesex", "Chelsea": "Suffolk", "Everett": "Middlesex",
    "Fitchburg": "Worcester", "Gloucester": "Essex", "Greenfield": "Franklin",
    "Lawrence": "Essex", "Lowell": "Middlesex", "Lynn": "Essex", "Malden": "Middlesex",
    "Medford": "Middlesex", "Methuen": "Essex", "New Bedford": "Bristol",
    "Newton": "Middlesex", "Northampton": "Hampshire", "Peabody": "Essex",
    "Pittsfield": "Berkshire", "Quincy": "Norfolk", "Revere": "Suffolk",
    "Salem": "Essex", "Somerville": "Middlesex", "Springfield": "Hampden",
    "Taunton": "Bristol", "Waltham": "Middlesex", "Watertown": "Middlesex",
    "Weymouth": "Norfolk", "Winthrop": "Suffolk", "Woburn": "Middlesex",
    "Worcester": "Worcester",
}

In [77]:
# Add county to your primary_df
primary_df['county'] = primary_df['town'].map(town_to_county)
primary_df.head(DISPLAY_ROWS)

,town,candidate,party,votes,county
0,Abington,All Others,Democratic,5,Plymouth
1,Abington,All Others,Green-Rainbow,0,Plymouth
2,Abington,All Others,Republican,2,Plymouth
3,Abington,Barack Obama,Democratic,824,Plymouth
4,Abington,Bill Richardson,Democratic,8,Plymouth
5,Abington,Blank Votes,Democratic,10,Plymouth
6,Abington,Blank Votes,Green-Rainbow,0,Plymouth
7,Abington,Blank Votes,Republican,1,Plymouth
8,Abington,Christopher J. Dodd,Democratic,1,Plymouth
9,Abington,Cynthia McKinney,Green-Rainbow,0,Plymouth


In [78]:
# Number of missing values in each column
primary_df.isna().sum()

town          0
candidate     0
party         0
votes         0
county       34
dtype: int64

There are 170 missing values in `county`. As how we define the mappings from town to county, all of those are TOTALS rows, which we can just drop.

In [79]:
# Observation with missing value in `county`
primary_df.loc[primary_df["county"].isna()]

,town,candidate,party,votes,county
9528,TOTALS,All Others,Democratic,3279,NaN
9529,TOTALS,All Others,Green-Rainbow,273,NaN
9530,TOTALS,All Others,Republican,1532,NaN
9531,TOTALS,Barack Obama,Democratic,511680,NaN
9532,TOTALS,Bill Richardson,Democratic,1846,NaN
9533,TOTALS,Blank Votes,Democratic,4841,NaN
9534,TOTALS,Blank Votes,Green-Rainbow,77,NaN
9535,TOTALS,Blank Votes,Republican,1447,NaN
9536,TOTALS,Christopher J. Dodd,Democratic,1120,NaN
9537,TOTALS,Cynthia McKinney,Green-Rainbow,474,NaN


In [80]:
# Drop rows with missing values in `county`
primary_df = primary_df[primary_df["county"].notna()]
primary_df.shape

(11444, 5)

In [81]:
# Different candidates in primary_df
primary_df["candidate"].value_counts()

candidate
All Others              1004
Blank Votes             1004
No Preference           1004
Total Votes Cast        1004
Christopher J. Dodd      351
Joseph R. Biden, Jr.     351
Tom Tancredo             351
Rudy Giuliani            351
Ron Paul                 351
Bill Richardson          351
Mitt Romney              351
Mike Huckabee            351
Mike Gravel              351
John McCain              351
Barack Obama             351
John Edwards             351
Hillary Clinton          351
Fred Thompson            351
Duncan Hunter            351
Dennis J. Kucinich       351
Cynthia McKinney         302
Kent Mesplay             302
Jared Ball               302
Ralph Nader              302
Elaine Brown             302
Kat Swift                302
Name: count, dtype: int64

Notice there are four values of candidates that we could drop from the list: "All Others", "Blank Votes", "No Preference", "Total Votes Cast".

In [82]:
# Drop rows where candidate is one of the four values above
primary_df = primary_df[~primary_df["candidate"].isin(
    ["All Others", "Blank Votes", "No Preference", "Total Votes Cast"]
)].reset_index(drop=True)

In [83]:
# Updated list of candidates in primary_df
primary_df["candidate"].value_counts()

candidate
Barack Obama            351
John Edwards            351
Rudy Giuliani           351
Ron Paul                351
Mitt Romney             351
Mike Huckabee           351
Mike Gravel             351
Joseph R. Biden, Jr.    351
Bill Richardson         351
John McCain             351
Hillary Clinton         351
Fred Thompson           351
Duncan Hunter           351
Dennis J. Kucinich      351
Christopher J. Dodd     351
Tom Tancredo            351
Jared Ball              302
Kat Swift               302
Kent Mesplay            302
Elaine Brown            302
Ralph Nader             302
Cynthia McKinney        302
Name: count, dtype: int64

In [84]:
# List out all the parties in the primary election data
primary_df["party"].value_counts()

party
Democratic       2808
Republican       2808
Green-Rainbow    1812
Name: count, dtype: int64

In [85]:
# Data type of each column in primary_df
primary_df.dtypes

town         object
candidate    object
party        object
votes         int64
county       object
dtype: object

In [86]:
# Drop the town column and rearrange
primary_df = primary_df[["county", "candidate", "party", "votes"]]

# Final look at the cleaned primary_df
primary_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Plymouth,Barack Obama,Democratic,824
1,Plymouth,Bill Richardson,Democratic,8
2,Plymouth,Christopher J. Dodd,Democratic,1
3,Plymouth,Cynthia McKinney,Green-Rainbow,0
4,Plymouth,Dennis J. Kucinich,Democratic,5
5,Plymouth,Duncan Hunter,Republican,1
6,Plymouth,Elaine Brown,Green-Rainbow,0
7,Plymouth,Fred Thompson,Republican,2
8,Plymouth,Hillary Clinton,Democratic,1915
9,Plymouth,Jared Ball,Green-Rainbow,0


In [87]:
# Shape after preprocessing
primary_df.shape

(7428, 4)

### b. General Election Dataset

In [88]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,town,ward,precinct,office,district,party,candidate,votes
0,Dracut,NaN,8,President,NaN,Green-Rainbow,McKinney and Clemente,1
1,Springfield,4,C,State Senate,Hampden,NaN,Blank Votes,129
2,Andover,NaN,8,U.S. Senate,NaN,NaN,Total Votes Cast,2175
3,Shrewsbury,NaN,8,U.S. Senate,NaN,Libertarian,Robert J. Underwood,34
4,Lowell,7,1,State House,18th Middlesex,Unenrolled,Kenneth J. Patrician,96
5,Worcester,1,4,President,NaN,Unenrolled,Nader and Gonzalez,16
6,Weymouth,NaN,2,President,NaN,NaN,Blank Votes,12
7,Worcester,3,3,Governor's Council,7th,Democratic,Thomas J. Foley,794
8,Lynn,6,1,President,NaN,NaN,All Others,7
9,Cambridge,7,3,President,NaN,NaN,No Preference,0


In [89]:
# Different values in 'office' column
general_df["office"].value_counts()

office
President             21730
U.S. Senate           13038
State House           10369
U.S. House             9836
State Senate           9500
Governor's Council     9039
Name: count, dtype: int64

In [90]:
# Only keep rows where 'office' is 'President'
general_df = general_df[general_df["office"] == "President"]
general_df.head(DISPLAY_ROWS)

,town,ward,precinct,office,district,party,candidate,votes
0,Dracut,NaN,8,President,NaN,Green-Rainbow,McKinney and Clemente,1
5,Worcester,1,4,President,NaN,Unenrolled,Nader and Gonzalez,16
6,Weymouth,NaN,2,President,NaN,NaN,Blank Votes,12
8,Lynn,6,1,President,NaN,NaN,All Others,7
9,Cambridge,7,3,President,NaN,NaN,No Preference,0
10,Medford,4,1,President,NaN,Green-Rainbow,McKinney and Clemente,3
17,Springfield,7,A,President,NaN,Constitution Party,Baldwin and Castle,3
19,Woburn,2,2,President,NaN,Libertarian,Barr and Root,2
20,Boston,12,5,President,NaN,NaN,Blank Votes,0
22,Hanover,NaN,2,President,NaN,Democratic,Obama and Biden,899


In [91]:
# General data shape when only considering President/VicePresident
general_df.shape

(21730, 8)

In [92]:
# Number of missing values in each column
general_df.isna().sum()

town             0
ward         11230
precinct        10
office           0
district     21730
party         8692
candidate        0
votes            0
dtype: int64

In [93]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the "district" column since it's all missing values
general_df = general_df.drop(columns=["office", "district"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,town,ward,precinct,party,candidate,votes
0,Dracut,NaN,8,Green-Rainbow,McKinney and Clemente,1
1,Worcester,1,4,Unenrolled,Nader and Gonzalez,16
2,Weymouth,NaN,2,NaN,Blank Votes,12
3,Lynn,6,1,NaN,All Others,7
4,Cambridge,7,3,NaN,No Preference,0
5,Medford,4,1,Green-Rainbow,McKinney and Clemente,3
6,Springfield,7,A,Constitution Party,Baldwin and Castle,3
7,Woburn,2,2,Libertarian,Barr and Root,2
8,Boston,12,5,NaN,Blank Votes,0
9,Hanover,NaN,2,Democratic,Obama and Biden,899


Again, we will groupby town to get the town-level vote count, then we will perform a cross-walk to map from town to county.

In [94]:
# Calculate town-level vote count
general_df = (
    general_df.groupby(['town', 'candidate', 'party'], as_index=False)['votes'].sum()
)

general_df.head(DISPLAY_ROWS)

,town,candidate,party,votes
0,Abington,Baldwin and Castle,Constitution Party,17
1,Abington,Barr and Root,Libertarian,34
2,Abington,McCain and Palin,Republican,3765
3,Abington,McKinney and Clemente,Green-Rainbow,10
4,Abington,Nader and Gonzalez,Unenrolled,75
5,Abington,Obama and Biden,Democratic,4094
6,Acton,Baldwin and Castle,Constitution Party,7
7,Acton,Barr and Root,Libertarian,71
8,Acton,McCain and Palin,Republican,3477
9,Acton,McKinney and Clemente,Green-Rainbow,18


In [95]:
# Add county to general_df
general_df['county'] = general_df['town'].map(town_to_county)
general_df.head(DISPLAY_ROWS)

,town,candidate,party,votes,county
0,Abington,Baldwin and Castle,Constitution Party,17,Plymouth
1,Abington,Barr and Root,Libertarian,34,Plymouth
2,Abington,McCain and Palin,Republican,3765,Plymouth
3,Abington,McKinney and Clemente,Green-Rainbow,10,Plymouth
4,Abington,Nader and Gonzalez,Unenrolled,75,Plymouth
5,Abington,Obama and Biden,Democratic,4094,Plymouth
6,Acton,Baldwin and Castle,Constitution Party,7,Middlesex
7,Acton,Barr and Root,Libertarian,71,Middlesex
8,Acton,McCain and Palin,Republican,3477,Middlesex
9,Acton,McKinney and Clemente,Green-Rainbow,18,Middlesex


In [97]:
# Number of missing values in each column
general_df.isna().sum()

town         0
candidate    0
party        0
votes        0
county       6
dtype: int64

There are 6 missing values in `county`. As how we define the mappings from town to county, all of those are TOTALS rows, which we can just drop.

In [99]:
# Observation with missing value in `county`
general_df.loc[general_df["county"].isna()]

,town,candidate,party,votes,county
1752,TOTALS,Baldwin and Castle,Constitution Party,4971,NaN
1753,TOTALS,Barr and Root,Libertarian,13189,NaN
1754,TOTALS,McCain and Palin,Republican,1108854,NaN
1755,TOTALS,McKinney and Clemente,Green-Rainbow,6550,NaN
1756,TOTALS,Nader and Gonzalez,Unenrolled,28841,NaN
1757,TOTALS,Obama and Biden,Democratic,1904097,NaN


In [100]:
# Drop rows with missing values in `county`
general_df = general_df[general_df["county"].notna()]
general_df.shape

(2106, 5)

In [104]:
# Different candidates in general_df
general_df["candidate"].value_counts()

candidate
Baldwin and Castle       351
Barr and Root            351
McCain and Palin         351
McKinney and Clemente    351
Nader and Gonzalez       351
Obama and Biden          351
Name: count, dtype: int64

Now, each row’s candidate value now contains two last names: presidential first, vice-presidential second, which is separated by an "and". We’ll split on the "and" and retain only the presidential name.

In [105]:
# Keep only the presidential candidate in the "candidate" column
general_df["candidate"] = (
    general_df["candidate"]
      .str.split(r"(?i)\s*(?:andf|and|&|/|/)\s*", n=1, expand=True)[0]
      .str.strip()
)

# Candidates in general_df
general_df["candidate"].value_counts()

candidate
Baldwin     351
Barr        351
McCain      351
McKinney    351
Nader       351
Obama       351
Name: count, dtype: int64

In [106]:
# List out all the parties in the primary election data
general_df["party"].value_counts()

party
Constitution Party    351
Libertarian           351
Republican            351
Green-Rainbow         351
Unenrolled            351
Democratic            351
Name: count, dtype: int64

In [107]:
# Data type of each column in general_df
general_df.dtypes

town         object
candidate    object
party        object
votes         int64
county       object
dtype: object

In [108]:
# Drop the town column and rearrange
general_df = general_df[["county", "candidate", "party", "votes"]]

# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Plymouth,Baldwin,Constitution Party,17
1,Plymouth,Barr,Libertarian,34
2,Plymouth,McCain,Republican,3765
3,Plymouth,McKinney,Green-Rainbow,10
4,Plymouth,Nader,Unenrolled,75
5,Plymouth,Obama,Democratic,4094
6,Middlesex,Baldwin,Constitution Party,7
7,Middlesex,Barr,Libertarian,71
8,Middlesex,McCain,Republican,3477
9,Middlesex,McKinney,Green-Rainbow,18


In [109]:
# Shape after preprocessing
general_df.shape

(2106, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [120]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Democratic -> dem, Republican -> rep
    """
    return(s.str.strip()
           .str.capitalize()
           .map({
                "Democratic"         : "dem",
                "Republican"         : "rep",
                "Libertarian"        : "lib",
                "Constitution party" : "cst",
                "Green-rainbow"      : "grn",
                "Unenrolled"         : "ind",               
                })
           .fillna(s.str.strip().str.lower()))      # For all others, they're all three-letter abbr, just need to lowercase

In [121]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [122]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [123]:
# Primary dataframe pivot
primary_pivot = pivot_wide(primary_df, prefix="pri")
primary_pivot.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_grn_BALL,...,pri_grn_NADER,pri_grn_SWIFT,pri_rep_GIULIANI,pri_rep_HUCKABEE,pri_rep_HUNTER,pri_rep_MCCAIN,pri_rep_PAUL,pri_rep_ROMNEY,pri_rep_TANCREDO,pri_rep_THOMPSON
0,Barnstable,105,24388,25,775,31,111,22352,83,0,...,37,4,162,902,4,10222,609,17633,8,56
1,Berkshire,59,15346,15,415,11,95,11844,29,1,...,28,7,56,588,6,3253,279,1969,2,27
2,Bristol,214,63412,76,1543,72,136,25817,98,3,...,41,3,185,1861,20,13237,913,15409,12,85
3,Dukes,5,1978,5,49,3,14,2890,5,1,...,6,0,22,65,0,560,35,619,0,1
4,Essex,347,83340,80,2401,104,280,51299,189,3,...,62,8,368,2295,37,27285,1673,35534,28,106
5,Franklin,16,7325,7,331,23,114,9198,22,2,...,31,4,22,396,1,2543,227,1343,1,7
6,Hampden,150,41950,89,1122,228,163,26911,101,4,...,36,3,207,1721,21,9941,743,13097,13,85
7,Hampshire,37,15613,19,492,40,199,18000,32,2,...,64,5,45,506,8,3869,317,3429,4,13
8,Middlesex,827,175048,492,5078,295,854,142823,516,8,...,192,11,621,3610,59,49449,3318,64344,39,185
9,Nantucket,2,833,2,35,2,1,1317,4,0,...,3,2,12,43,0,428,27,454,1,0


In [124]:
# Primary dataframe shape after pivot
primary_pivot.shape

(14, 23)

In [125]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN
0,Barnstable,195,74264,215,1071,489,55694
1,Berkshire,169,49558,209,855,248,14876
2,Bristol,579,146861,435,2490,913,90531
3,Dukes,14,7913,29,93,37,2442
4,Essex,518,208976,603,3055,1604,137129
5,Franklin,84,27919,165,494,182,9545
6,Hampden,367,121454,444,2588,715,71350
7,Hampshire,110,56869,344,998,335,20618
8,Middlesex,1014,464484,1614,6091,3575,245766
9,Nantucket,8,4073,11,49,23,1863


In [126]:
# General dataframe shape after pivot
general_pivot.shape

(14, 7)

## 4. Merge Dataframes

Before merging, we verify that county names match across primary and general:

In [117]:
# Check if county names match between primary_df and general_df
primary_counties = set(primary_pivot["county"].unique())
general_counties = set(general_pivot["county"].unique())
common_counties = primary_counties.intersection(general_counties)
print(f"Number of common counties: {len(common_counties)} out of {len(primary_counties)}")

Number of common counties: 14 out of 14


Great. Since we know that all counties name are matched, we don't need to perform further data preprocessing to match the county names. Thus, we can now merge them:

In [118]:
# Merge primary and general dataframes on 'county'
merged_df = primary_pivot.merge(general_pivot, on="county", how="inner").fillna(0)    # There should be no missing values to fill with 0
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_grn_BALL,...,pri_rep_PAUL,pri_rep_ROMNEY,pri_rep_TANCREDO,pri_rep_THOMPSON,gen_con_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN
0,Barnstable,105,24388,25,775,31,111,22352,83,0,...,609,17633,8,56,195,74264,215,1071,489,55694
1,Berkshire,59,15346,15,415,11,95,11844,29,1,...,279,1969,2,27,169,49558,209,855,248,14876
2,Bristol,214,63412,76,1543,72,136,25817,98,3,...,913,15409,12,85,579,146861,435,2490,913,90531
3,Dukes,5,1978,5,49,3,14,2890,5,1,...,35,619,0,1,14,7913,29,93,37,2442
4,Essex,347,83340,80,2401,104,280,51299,189,3,...,1673,35534,28,106,518,208976,603,3055,1604,137129
5,Franklin,16,7325,7,331,23,114,9198,22,2,...,227,1343,1,7,84,27919,165,494,182,9545
6,Hampden,150,41950,89,1122,228,163,26911,101,4,...,743,13097,13,85,367,121454,444,2588,715,71350
7,Hampshire,37,15613,19,492,40,199,18000,32,2,...,317,3429,4,13,110,56869,344,998,335,20618
8,Middlesex,827,175048,492,5078,295,854,142823,516,8,...,3318,64344,39,185,1014,464484,1614,6091,3575,245766
9,Nantucket,2,833,2,35,2,1,1317,4,0,...,27,454,1,0,8,4073,11,49,23,1863


In [119]:
# Statistics check on merged dataframe 
merged_df.describe()

,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_grn_BALL,pri_grn_BROWN,...,pri_rep_PAUL,pri_rep_ROMNEY,pri_rep_TANCREDO,pri_rep_THOMPSON,gen_con_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN
count,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,...,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000
mean,229.714286,50370.357143,80.000000,1435.785714,104.500000,213.714286,36548.571429,131.857143,3.000000,2.714286,...,946.500000,18278.000000,10.928571,65.428571,355.071429,136006.928571,467.857143,2060.071429,942.071429,79203.857143
std,236.538313,46982.042595,123.828666,1341.340901,112.178258,209.357416,36511.053032,137.407231,2.660249,3.517617,...,905.116716,18503.159535,10.964729,55.181817,294.558200,120965.380925,407.671598,1684.503920,948.257062,71464.816775
min,2.000000,833.000000,2.000000,35.000000,2.000000,1.000000,1317.000000,4.000000,0.000000,0.000000,...,27.000000,454.000000,0.000000,0.000000,8.000000,4073.000000,11.000000,49.000000,23.000000,1863.000000
25%,42.500000,15412.750000,16.000000,434.250000,25.000000,111.750000,13383.000000,29.750000,1.000000,1.000000,...,288.500000,2334.000000,2.500000,16.500000,124.750000,51385.750000,210.500000,890.750000,269.750000,16311.500000
50%,182.000000,46222.500000,64.500000,1332.500000,68.000000,151.000000,26364.000000,99.500000,2.500000,1.500000,...,795.000000,14253.000000,9.000000,70.000000,331.500000,126635.500000,411.000000,1972.000000,803.000000,64272.000000
75%,332.250000,74447.250000,79.000000,2023.500000,119.000000,276.500000,49890.500000,179.000000,3.750000,3.000000,...,1272.250000,31446.750000,12.750000,94.000000,510.750000,201749.000000,597.250000,2870.750000,1286.250000,130856.750000
max,827.000000,175048.000000,492.000000,5078.000000,356.000000,854.000000,142823.000000,516.000000,8.000000,14.000000,...,3318.000000,64344.000000,39.000000,185.000000,1014.000000,464484.000000,1614.000000,6091.000000,3575.000000,245766.000000


Now, we will add party totals columns: 

- Primary totals:
    * `rep_primary_total` = sum of all `pri_rep_*` columns
    * `dem_primary_total` = sum of all `pri_dem_*` columns
    * `grn_primary_total` = sum of all `pri_grn_*` columns


- General totals:
    * `rep_general_total` = sum of all `gen_rep_*` columns
    * `dem_general_total` = sum of all `gen_dem_*` columns
    * `lib_general_total` = sum of all `gen_lib_*` columns
    * `cst_general_total` = sum of all `gen_cst_*` columns
    * `grn_general_total` = sum of all `gen_grn_*` columns
    * `ind_general_total` = sum of all `gen_ind_*` columns

In [127]:
# Add party totals for primary election
rep_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_rep_")]
dem_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_dem_")]
grn_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_grn_")]

merged_df["rep_primary_total"] = merged_df[rep_primary_cols].sum(axis=1) if rep_primary_cols else 0
merged_df["dem_primary_total"] = merged_df[dem_primary_cols].sum(axis=1) if dem_primary_cols else 0
merged_df["grn_primary_total"] = merged_df[grn_primary_cols].sum(axis=1) if grn_primary_cols else 0

In [128]:
# Add party totals for general election
rep_general_cols   = [c for c in merged_df.columns if c.startswith("gen_rep_")]
dem_general_cols   = [c for c in merged_df.columns if c.startswith("gen_dem_")]
lib_general_cols   = [c for c in merged_df.columns if c.startswith("gen_lib_")]
cst_general_cols   = [c for c in merged_df.columns if c.startswith("gen_cst_")]
grn_general_cols   = [c for c in merged_df.columns if c.startswith("gen_grn_")]
ind_general_cols   = [c for c in merged_df.columns if c.startswith("gen_ind_")]

merged_df["rep_general_total"] = merged_df[rep_general_cols].sum(axis=1) if rep_general_cols else 0
merged_df["dem_general_total"] = merged_df[dem_general_cols].sum(axis=1) if dem_general_cols else 0
merged_df["lib_general_total"] = merged_df[lib_general_cols].sum(axis=1) if lib_general_cols else 0
merged_df["cst_general_total"] = merged_df[cst_general_cols].sum(axis=1) if cst_general_cols else 0
merged_df["grn_general_total"] = merged_df[grn_general_cols].sum(axis=1) if grn_general_cols else 0
merged_df["ind_general_total"] = merged_df[ind_general_cols].sum(axis=1) if ind_general_cols else 0

In [129]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned dataframe:")
merged_df.columns

Final columns in the cleaned dataframe:


Index(['county', 'pri_dem_BIDEN', 'pri_dem_CLINTON', 'pri_dem_DODD',
       'pri_dem_EDWARDS', 'pri_dem_GRAVEL', 'pri_dem_KUCINICH',
       'pri_dem_OBAMA', 'pri_dem_RICHARDSON', 'pri_grn_BALL', 'pri_grn_BROWN',
       'pri_grn_MCKINNEY', 'pri_grn_MESPLAY', 'pri_grn_NADER', 'pri_grn_SWIFT',
       'pri_rep_GIULIANI', 'pri_rep_HUCKABEE', 'pri_rep_HUNTER',
       'pri_rep_MCCAIN', 'pri_rep_PAUL', 'pri_rep_ROMNEY', 'pri_rep_TANCREDO',
       'pri_rep_THOMPSON', 'gen_con_BALDWIN', 'gen_dem_OBAMA',
       'gen_grn_MCKINNEY', 'gen_ind_NADER', 'gen_lib_BARR', 'gen_rep_MCCAIN',
       'rep_primary_total', 'dem_primary_total', 'grn_primary_total',
       'rep_general_total', 'dem_general_total', 'lib_general_total',
       'cst_general_total', 'grn_general_total', 'ind_general_total'],
      dtype='object')

In [130]:
# Preview merged dataframe with totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_grn_BALL,...,gen_rep_MCCAIN,rep_primary_total,dem_primary_total,grn_primary_total,rep_general_total,dem_general_total,lib_general_total,cst_general_total,grn_general_total,ind_general_total
0,Barnstable,105,24388,25,775,31,111,22352,83,0,...,55694,29596,47870,49,55694,74264,489,0,215,1071
1,Berkshire,59,15346,15,415,11,95,11844,29,1,...,14876,6180,27814,64,14876,49558,248,0,209,855
2,Bristol,214,63412,76,1543,72,136,25817,98,3,...,90531,31722,91368,55,90531,146861,913,0,435,2490
3,Dukes,5,1978,5,49,3,14,2890,5,1,...,2442,1302,4949,9,2442,7913,37,0,29,93
4,Essex,347,83340,80,2401,104,280,51299,189,3,...,137129,67326,138040,102,137129,208976,1604,0,603,3055
5,Franklin,16,7325,7,331,23,114,9198,22,2,...,9545,4540,17036,70,9545,27919,182,0,165,494
6,Hampden,150,41950,89,1122,228,163,26911,101,4,...,71350,25828,70714,67,71350,121454,715,0,444,2588
7,Hampshire,37,15613,19,492,40,199,18000,32,2,...,20618,8191,34432,134,20618,56869,335,0,344,998
8,Middlesex,827,175048,492,5078,295,854,142823,516,8,...,245766,121625,325933,392,245766,464484,3575,0,1614,6091
9,Nantucket,2,833,2,35,2,1,1317,4,0,...,1863,965,2196,5,1863,4073,23,0,11,49


In [131]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(OUTPUT_PATH + "MA.csv", index=False)